# <center>Procesamiento V2 Seguro para Arboles</center>


Este notebook genera una nueva variante de preprocesamiento pensada para modelos de arboles. Cumple el contrato pedido para esta iteracion: `-999 -> NaN` apenas se carga, split train/valid antes de ajustar cualquier transformacion, `PRI_jet_num` tratada como categorica, indicadores de missing y guardado del pipeline ajustado junto con los CSV procesados.


## Imports y utilidades


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_try_dir(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        direct_candidate = candidate / "higgs_preprocessing_v2.py"
        nested_candidate = candidate / "10.try" / "higgs_preprocessing_v2.py"
        if direct_candidate.exists():
            return candidate
        if nested_candidate.exists():
            return candidate / "10.try"
    raise FileNotFoundError("No se pudo ubicar 10.try/higgs_preprocessing_v2.py")


try_dir = find_try_dir()
if str(try_dir) not in sys.path:
    sys.path.insert(0, str(try_dir))

from higgs_preprocessing_v2 import (
    VARIANT_DEFINITIONS,
    get_higgs_data_dir,
    run_preprocessing_variant,
    save_preprocessing_artifacts,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


## Variante elegida
- No descarta columnas por missing.
- No imputa numericas: los `NaN` se conservan para que el modelo de arboles los use directamente.
- Agrega flags `is_missing_*` para columnas con faltantes en train.
- Hace one-hot encoding de `PRI_jet_num`.


In [2]:
variant_key = "tree_safe"
variant = VARIANT_DEFINITIONS[variant_key]
artifacts = run_preprocessing_variant(variant_key=variant_key)
output_dir = get_higgs_data_dir() / "processed" / variant.output_dir_name

print(f"Variante: {variant.title}")
print(f"Directorio de salida esperado: {output_dir}")
print("Descripcion:")
print(variant.description)


Variante: Procesamiento V2 - Seguro para Arboles
Directorio de salida esperado: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe
Descripcion:
Mantiene los NaN para modelos de arboles, agrega indicadores de missing, trata PRI_jet_num como categorica y evita imputacion o escalado numerico.


## Resumen del procesamiento


In [3]:
metadata = artifacts["metadata"]

print("Resumen del split temprano:")
display(artifacts["split_summary_df"].round(4))

print("Missingness en train crudo luego del reemplazo -999 -> NaN:")
display(artifacts["train_missing_summary_df"].head(15))

print("Metadata principal del procesamiento:")
for key, value in metadata.items():
    print(f"- {key}: {value}")

print("Primeras filas del train procesado:")
display(artifacts["X_train_processed"].head())
print("Etiquetas de train:")
display(artifacts["y_train"].head().to_frame(name="Label"))
print("Columnas con flags de missing:")
display(pd.DataFrame({"feature": artifacts["missing_indicator_source_columns"]}).head(20))


Resumen del split temprano:


,split,rows,class_b,class_s,weight_sum
0,train,225000,147900.0,77100.0,370822.7398
1,valid,25000,16433.0,8567.0,40869.0961
2,test_external,550000,NaN,NaN,NaN


Missingness en train crudo luego del reemplazo -999 -> NaN:


,feature,missing_count,missing_pct
0,DER_deltaeta_jet_jet,159718,70.99
1,DER_lep_eta_centrality,159718,70.99
2,DER_mass_jet_jet,159718,70.99
3,DER_prodeta_jet_jet,159718,70.99
4,PRI_jet_subleading_eta,159718,70.99
5,PRI_jet_subleading_phi,159718,70.99
6,PRI_jet_subleading_pt,159718,70.99
7,PRI_jet_leading_eta,89987,39.99
8,PRI_jet_leading_phi,89987,39.99
9,PRI_jet_leading_pt,89987,39.99


Metadata principal del procesamiento:
- variant_key: tree_safe
- variant_title: Procesamiento V2 - Seguro para Arboles
- description: Mantiene los NaN para modelos de arboles, agrega indicadores de missing, trata PRI_jet_num como categorica y evita imputacion o escalado numerico.
- valid_size: 0.1
- random_state: 42
- raw_feature_count: 30
- processed_feature_count: 44
- processed_train_nan_count: 1422313
- processed_valid_nan_count: 157739
- processed_test_nan_count: 3473394
- expects_missing_after_processing: True
Primeras filas del train procesado:


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,DER_pt_ratio_lep_tau,DER_met_phi_centrality,DER_lep_eta_centrality,PRI_tau_pt,PRI_tau_eta,PRI_tau_phi,PRI_lep_pt,PRI_lep_eta,PRI_lep_phi,PRI_met,PRI_met_phi,PRI_met_sumet,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,is_missing_DER_mass_MMC,is_missing_DER_deltaeta_jet_jet,is_missing_DER_mass_jet_jet,is_missing_DER_prodeta_jet_jet,is_missing_DER_lep_eta_centrality,is_missing_PRI_jet_leading_pt,is_missing_PRI_jet_leading_eta,is_missing_PRI_jet_leading_phi,is_missing_PRI_jet_subleading_pt,is_missing_PRI_jet_subleading_eta,is_missing_PRI_jet_subleading_phi,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,109.492996,40.445999,81.954002,37.280998,NaN,NaN,NaN,2.906,37.280998,86.779999,0.525,-1.406,NaN,56.905998,0.401,2.182,29.874001,0.324,-0.723,15.662000,-3.139,155.645004,NaN,NaN,NaN,NaN,NaN,NaN,-0.000000,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
1,NaN,71.706001,104.612999,1.741000,NaN,NaN,NaN,2.618,1.741000,93.132004,1.041,-1.413,NaN,45.625000,0.830,-2.159,47.507000,2.054,1.810,36.578999,-0.261,115.133003,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
2,106.320000,7.259000,57.109001,91.891998,NaN,NaN,NaN,1.971,25.091000,176.307999,1.003,1.154,NaN,31.552000,2.059,2.421,31.656000,1.125,-2.127,56.681000,-2.298,309.502014,113.099998,-2.667,0.686,NaN,NaN,NaN,113.099998,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
3,NaN,81.124001,91.842003,46.893002,NaN,NaN,NaN,2.862,1.888000,127.359001,0.761,-1.408,NaN,45.945999,0.765,-1.780,34.952999,1.932,0.833,60.183998,3.003,150.822006,46.459000,1.779,0.271,NaN,NaN,NaN,46.459000,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0
4,88.685997,46.839001,59.231998,121.435997,3.813,637.747009,-3.345,1.614,13.441000,296.751007,2.641,1.378,0.288,23.315001,-0.651,-1.853,61.580002,-1.587,-0.539,54.285999,-1.373,232.520996,157.104004,-1.368,2.211,54.752998,2.445,-0.481,211.856995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


Etiquetas de train:


,Label
0,s
1,b
2,s
3,b
4,s


Columnas con flags de missing:


,feature
0,DER_mass_MMC
1,DER_deltaeta_jet_jet
2,DER_mass_jet_jet
3,DER_prodeta_jet_jet
4,DER_lep_eta_centrality
5,PRI_jet_leading_pt
6,PRI_jet_leading_eta
7,PRI_jet_leading_phi
8,PRI_jet_subleading_pt
9,PRI_jet_subleading_eta


## Guardado de artefactos
Se persisten los `csv` procesados del split train/valid/test, los IDs, los pesos, la metadata del split y el `joblib` con el pipeline ajustado.


In [4]:
saved_paths = save_preprocessing_artifacts(artifacts, output_dir=output_dir)

print(f"Directorio de salida: {output_dir}")
print(f"Cantidad de artefactos guardados: {len(saved_paths)}")
for path in saved_paths:
    print(f"- {path}")


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe
Cantidad de artefactos guardados: 19
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/X_valid_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/y_valid.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/valid_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/v2_tree_safe/valid_event_id.csv
- /home/sant